In [1]:
import json
from pathlib import Path
from imgnet.collections.store import IndexedDatasets
from imgnet.collections.source import S3Source
from imgnet.download import download_collection
from imgtools.nifti.crawl import Crawler
from rich import print

/home/gpudual/bhklab/josh/med-image_index/.pixi/envs/default/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
indexed_datasets = IndexedDatasets("indexed_datasets") # This will download the latest version of the indexed_datasets if you dont provide a path
print(indexed_datasets.collections[:10])

/home/gpudual/bhklab/josh/med-image_index/notebooks/indexed_datasets

[
    '4D-Lung',
    'ACRIN-6698',
    'ACRIN-Contralateral-Breast-MR',
    'ACRIN-FLT-Breast',
    'ACRIN-NSCLC-FDG-PET',
    'Adrenal-ACC-Ki67-Seg',
    'Advanced-MRI-Breast-Lesions',
    'Anti-PD-1_Lung',
    'B-mode-and-CEUS-Liver',
    'BREAST-DIAGNOSIS'
]

In [47]:
source = S3Source(
    file_type="nifti",
    source="s3",
    bucket_name="msd-for-monai",
    description="The Medical Segmentation Decathlon is a collection of 3D CT and MRI scans with expert-labeled segmentations across ten different medical tasks, such as organs and tumors. It’s widely used to evaluate how well models generalize across different anatomies and imaging types in medical image segmentation."
)


In [48]:
dataset_index_path = (indexed_datasets.imgtools_path / "MedicalDecathalon")
dataset_index_path.mkdir(parents=True, exist_ok=True)
with open(dataset_index_path / "source.json", "w") as f:
    json.dump(source.model_dump(mode="json"), f, indent=2)

In [15]:
print(indexed_datasets.source_config("MedicalDecathalon"))

S3Source(
    file_type=<FileType.NIFTI: 'nifti'>,
    source='s3',
    bucket_name='msd-for-monai',
    filenames=None,
    post_download=['unzip']
)

In [6]:
temp_data_path = Path("temp_data") / "MedicalDecathalon"
temp_data_path.mkdir(parents=True, exist_ok=True)
download_collection(temp_data_path, source)

11:51:49 [info     ] Downloading all 11 files from S3 bucket msd-for-monai [imgnet] call=dispatcher._download_s3:54
         [info     ] Downloading file 'Task01_BrainTumour.tar' from S3 bucket msd-for-monai [imgnet] call=dispatcher._download_s3:64
Task01_BrainTumour.tar: 100%|██████████| 7.61G/7.61G [26:21<00:00, 4.81MB/s]   
12:18:11 [info     ] Downloading file 'Task02_Heart.tar' from S3 bucket msd-for-monai [imgnet] call=dispatcher._download_s3:64
Task02_Heart.tar: 100%|██████████| 456M/456M [01:44<00:00, 4.35MB/s]  
12:19:56 [info     ] Downloading file 'Task03_Liver.tar' from S3 bucket msd-for-monai [imgnet] call=dispatcher._download_s3:64
Task03_Liver.tar: 100%|██████████| 28.9G/28.9G [1:52:24<00:00, 4.29MB/s]  
14:12:20 [info     ] Downloading file 'Task04_Hippocampus.tar' from S3 bucket msd-for-monai [imgnet] call=dispatcher._download_s3:64
Task04_Hippocampus.tar: 100%|██████████| 28.4M/28.4M [00:06<00:00, 4.67MB/s]
14:12:27 [info     ] Downloading file 'Task05_Prostate.tar' f

In [6]:
from pathlib import Path

temp_data_path = Path("temp_data") / "MedicalDecathalon"
# Remove weird files
for p in temp_data_path.rglob("._*"):
    p.unlink()

In [10]:
import imgtools.loggers
import logging
import os

cpu_count = os.cpu_count()

imgtools.loggers.get_logger("imgtools").setLevel(logging.INFO)

crawler = Crawler(
    nifti_dir=temp_data_path,
    scan_name_pattern="{BodyPartExamined}/images{split}/{ImageID}.nii.gz",
    mask_name_pattern="{BodyPartExamined}/labels{split}/{ImageID}.nii.gz",
    output_dir=indexed_datasets.imgtools_path,
    dataset_name="MedicalDecathalon",
    n_jobs=cpu_count-4,
    force=True,
    deep=True,
)
crawler.crawl()


12:27:16 [info     ] Starting NIFTI crawl.          [imgtools] nifti_dir=PosixPath('temp_data/MedicalDecathalon') output_dir=PosixPath('indexed_datasets/.imgtools') dataset_name=MedicalDecathalon call=crawler.crawl:78
         [info     ] Found 4369 NIfTI files in /home/gpudual/bhklab/josh/med-image_index/notebooks/temp_data/MedicalDecathalon. [imgtools] call=parse_niftis.parse_nifti_dir:426
         [info     ] Using shared keys: ['BodyPartExamined', 'split', 'ImageID'] for reference_id, this will be used to link masks to their referenced scans [imgtools] call=parse_niftis.parse_nifti_dir:447
Parsing 4369 files:   0%|          | 0/4369 [00:00<?, ?it/s]12:27:17 [error    ] Error reading image /home/gpudual/bhklab/josh/med-image_index/notebooks/temp_data/MedicalDecathalon/Task01_BrainTumour/imagesTr/BRATS_017.nii.gz: Exception thrown in SimpleITK StatisticsImageFilter_Execute: /work/src/Code/Common/include/sitkMemberFunctionFactory.hxx:139:
sitk::ERROR: Pixel type: 32-bit float is not s

KeyboardInterrupt: 

In [28]:
import pandas as pd

index_df = indexed_datasets.index("MedicalDecathalon")
index_df.head()

,BodyPartExamined,split,ImageID,filepath,file_type,reference_id,class,hash,size,ndim,...,mask.bbox.max_coord,mask.feret_diameter,mask.roundness,mask.flatness,mask.elongation,mask.equivalent_spherical_radius,mask.equivalent_spherical_perimeter,mask.equivalent_ellipsoid_diameters,mask.volume_count,reference_scan
0,Task01_BrainTumour,Tr,BRATS_001,Task01_BrainTumour/imagesTr/BRATS_001.nii.gz,scan,Task01_BrainTumour_Tr_BRATS_001,MedImage,3d3c950a77051659e04b693871852b00f4ba7304,"(240, 240, 620)",3,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Task01_BrainTumour,Tr,BRATS_002,Task01_BrainTumour/imagesTr/BRATS_002.nii.gz,scan,Task01_BrainTumour_Tr_BRATS_002,MedImage,796d64011be8fcaf66f5ac8e26f27cb63847f882,"(240, 240, 620)",3,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Task01_BrainTumour,Tr,BRATS_003,Task01_BrainTumour/imagesTr/BRATS_003.nii.gz,scan,Task01_BrainTumour_Tr_BRATS_003,MedImage,fcbae1fba75fe6457940ac8beb4c9dcfa2dc29be,"(240, 240, 620)",3,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Task01_BrainTumour,Tr,BRATS_004,Task01_BrainTumour/imagesTr/BRATS_004.nii.gz,scan,Task01_BrainTumour_Tr_BRATS_004,MedImage,d54bccbe4ca6df61b4c7390011a496c11aebec3c,"(240, 240, 620)",3,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Task01_BrainTumour,Tr,BRATS_005,Task01_BrainTumour/imagesTr/BRATS_005.nii.gz,scan,Task01_BrainTumour_Tr_BRATS_005,MedImage,2f0ceff49b518db9a4d96c11454ade1ea1516675,"(240, 240, 620)",3,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [36]:
temp_data_path = Path("temp_data") / "MedicalDecathalon"
for task in index_df["BodyPartExamined"].unique():
    dataset_json = temp_data_path / task / "dataset.json"
    if dataset_json.exists():
        with open(dataset_json, "r") as f:
            dataset = json.load(f)

    labels = dataset["labels"]
    modalities = dataset["modality"]
    labels = ", ".join([v for _, v in labels.items() if v != "background"])
    modalities = ", ".join([v for _, v in modalities.items()])
    if modalities == "MRI":
        modalities = "MR"
    index_df.loc[index_df["BodyPartExamined"] == task, "ROINames"] = labels
    index_df.loc[index_df["BodyPartExamined"] == task, "Modality"] = modalities


In [39]:
index_df["BodyPartExamined"] = (
    index_df["BodyPartExamined"].astype(str).str.split("_").str[1]
)



In [40]:
index_df["BodyPartExamined"].value_counts()

BodyPartExamined
BrainTumour      1234
HepaticVessel     746
Pancreas          701
Hippocampus       650
Liver             332
Colon             316
Lung              158
Spleen            102
Prostate           80
Heart              50
Name: count, dtype: int64

In [42]:
index_df.to_csv(indexed_datasets.imgtools_path / "MedicalDecathalon" / "index.csv", index=False)

In [44]:
index_df = indexed_datasets.index("MedicalDecathalon")
index_df

,BodyPartExamined,split,ImageID,filepath,file_type,reference_id,class,hash,size,ndim,...,mask.roundness,mask.flatness,mask.elongation,mask.equivalent_spherical_radius,mask.equivalent_spherical_perimeter,mask.equivalent_ellipsoid_diameters,mask.volume_count,reference_scan,ROINames,Modality
0,BrainTumour,Tr,BRATS_001,Task01_BrainTumour/imagesTr/BRATS_001.nii.gz,scan,Task01_BrainTumour_Tr_BRATS_001,MedImage,3d3c950a77051659e04b693871852b00f4ba7304,"(240, 240, 620)",3,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"edema, non-enhancing tumor, enhancing tumour","FLAIR, T1w, t1gd, T2w"
1,BrainTumour,Tr,BRATS_002,Task01_BrainTumour/imagesTr/BRATS_002.nii.gz,scan,Task01_BrainTumour_Tr_BRATS_002,MedImage,796d64011be8fcaf66f5ac8e26f27cb63847f882,"(240, 240, 620)",3,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"edema, non-enhancing tumor, enhancing tumour","FLAIR, T1w, t1gd, T2w"
2,BrainTumour,Tr,BRATS_003,Task01_BrainTumour/imagesTr/BRATS_003.nii.gz,scan,Task01_BrainTumour_Tr_BRATS_003,MedImage,fcbae1fba75fe6457940ac8beb4c9dcfa2dc29be,"(240, 240, 620)",3,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"edema, non-enhancing tumor, enhancing tumour","FLAIR, T1w, t1gd, T2w"
3,BrainTumour,Tr,BRATS_004,Task01_BrainTumour/imagesTr/BRATS_004.nii.gz,scan,Task01_BrainTumour_Tr_BRATS_004,MedImage,d54bccbe4ca6df61b4c7390011a496c11aebec3c,"(240, 240, 620)",3,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"edema, non-enhancing tumor, enhancing tumour","FLAIR, T1w, t1gd, T2w"
4,BrainTumour,Tr,BRATS_005,Task01_BrainTumour/imagesTr/BRATS_005.nii.gz,scan,Task01_BrainTumour_Tr_BRATS_005,MedImage,2f0ceff49b518db9a4d96c11454ade1ea1516675,"(240, 240, 620)",3,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"edema, non-enhancing tumor, enhancing tumour","FLAIR, T1w, t1gd, T2w"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4364,Colon,Tr,colon_215,Task10_Colon/labelsTr/colon_215.nii.gz,mask,Task10_Colon_Tr_colon_215,Mask,5f4eee388d796a497e4841cf951e4105a3375e19,"(512, 512, 92)",3,...,0.805459,1.865516,1.293768,11.920381,1785.624593,"(14.437834281854327, 26.934006295205407, 34.84...",1.0,Task10_Colon/imagesTr/colon_215.nii.gz,colon cancer primaries,CT
4365,Colon,Tr,colon_216,Task10_Colon/labelsTr/colon_216.nii.gz,mask,Task10_Colon_Tr_colon_216,Mask,9abb6686934896f2daf7aa2e3a1e900e4312d06f,"(512, 512, 90)",3,...,0.507679,1.153354,1.660490,14.843588,2768.774925,"(22.795466117879528, 26.291231368785436, 43.65...",1.0,Task10_Colon/imagesTr/colon_216.nii.gz,colon cancer primaries,CT
4366,Colon,Tr,colon_217,Task10_Colon/labelsTr/colon_217.nii.gz,mask,Task10_Colon_Tr_colon_217,Mask,e90c69f2cf0a1c76ffafdeec68248a00365e8946,"(512, 512, 96)",3,...,0.565408,1.562260,1.304591,12.836191,2070.533167,"(17.450414466292635, 27.262092487174424, 35.56...",1.0,Task10_Colon/imagesTr/colon_217.nii.gz,colon cancer primaries,CT
4367,Colon,Tr,colon_218,Task10_Colon/labelsTr/colon_218.nii.gz,mask,Task10_Colon_Tr_colon_218,Mask,8977e7e84fa67c9b4377ee9af743b51281795c17,"(512, 512, 90)",3,...,0.535958,1.581470,1.457455,15.005894,2829.655908,"(19.500820073404867, 30.839959961567512, 44.94...",1.0,Task10_Colon/imagesTr/colon_218.nii.gz,colon cancer primaries,CT


In [1]:
from imgnet.collections.store import IndexedDatasets

indexed_datasets = IndexedDatasets()

/home/gpudual/bhklab/josh/med-image_index/.pixi/envs/default/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
14:20:01 [info     ] Indexed datasets path: /home/gpudual/.local/share/med-imagenet/indexed_datasets [imgnet] call=store.__init__:67


In [2]:
from imgindex.model import validate_index

index = indexed_datasets.index("MedicalDecathalon")
validate_index(index, "nifti", lazy=True)

Invalid NIFTI index: {
    "SCHEMA": {
        "COLUMN_NOT_IN_DATAFRAME": [
            {
                "schema": "NiftiIndex",
                "column": "NiftiIndex",
                "check": "column_in_dataframe",
                "error": "column 'SampleID' not in dataframe. Columns in dataframe: ['BodyPartExamined', 'split', 'ImageID', 'filepath', 'file_type', 'reference_id', 'class', 'hash', 'size', 'ndim', 'nvoxels', 'spacing', 'origin', 'direction', 'min', 'max', 'sum', 'mean', 'std', 'variance', 'dtype_str', 'dtype_numpy', 'mask.bbox.size', 'mask.bbox.min_coord', 'mask.bbox.max_coord', 'mask.feret_diameter', 'mask.roundness', 'mask.flatness', 'mask.elongation', 'mask.equivalent_spherical_radius', 'mask.equivalent_spherical_perimeter', 'mask.equivalent_ellipsoid_diameters', 'mask.volume_count', 'reference_scan', 'ROINames', 'Modality']"
            }
        ]
    },
    "DATA": {
        "DATAFRAME_CHECK": [
            {
                "schema": "NiftiIndex",
               

In [3]:
index.rename(columns={"reference_id": "SampleID"}, inplace=True)

In [4]:
index["Modality"].value_counts()

Modality
CT                       2355
FLAIR, T1w, t1gd, T2w    1234
MR                        700
T2, ADC                    80
Name: count, dtype: int64

In [6]:
index["SeriesDescription"] = ""
for i, row in index.iterrows():
    if not row["Modality"] in ["MR", "CT"]:
        index.loc[i, "SeriesDescription"] = row["Modality"]
        index.loc[i, "Modality"] = "MR"





In [9]:
index

,BodyPartExamined,split,ImageID,filepath,file_type,SampleID,class,hash,size,ndim,...,mask.flatness,mask.elongation,mask.equivalent_spherical_radius,mask.equivalent_spherical_perimeter,mask.equivalent_ellipsoid_diameters,mask.volume_count,reference_scan,ROINames,Modality,SeriesDescription
0,BrainTumour,Tr,BRATS_001,Task01_BrainTumour/imagesTr/BRATS_001.nii.gz,scan,Task01_BrainTumour_Tr_BRATS_001,MedImage,3d3c950a77051659e04b693871852b00f4ba7304,"(240, 240, 620)",3,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"edema, non-enhancing tumor, enhancing tumour",MR,"FLAIR, T1w, t1gd, T2w"
1,BrainTumour,Tr,BRATS_002,Task01_BrainTumour/imagesTr/BRATS_002.nii.gz,scan,Task01_BrainTumour_Tr_BRATS_002,MedImage,796d64011be8fcaf66f5ac8e26f27cb63847f882,"(240, 240, 620)",3,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"edema, non-enhancing tumor, enhancing tumour",MR,"FLAIR, T1w, t1gd, T2w"
2,BrainTumour,Tr,BRATS_003,Task01_BrainTumour/imagesTr/BRATS_003.nii.gz,scan,Task01_BrainTumour_Tr_BRATS_003,MedImage,fcbae1fba75fe6457940ac8beb4c9dcfa2dc29be,"(240, 240, 620)",3,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"edema, non-enhancing tumor, enhancing tumour",MR,"FLAIR, T1w, t1gd, T2w"
3,BrainTumour,Tr,BRATS_004,Task01_BrainTumour/imagesTr/BRATS_004.nii.gz,scan,Task01_BrainTumour_Tr_BRATS_004,MedImage,d54bccbe4ca6df61b4c7390011a496c11aebec3c,"(240, 240, 620)",3,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"edema, non-enhancing tumor, enhancing tumour",MR,"FLAIR, T1w, t1gd, T2w"
4,BrainTumour,Tr,BRATS_005,Task01_BrainTumour/imagesTr/BRATS_005.nii.gz,scan,Task01_BrainTumour_Tr_BRATS_005,MedImage,2f0ceff49b518db9a4d96c11454ade1ea1516675,"(240, 240, 620)",3,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"edema, non-enhancing tumor, enhancing tumour",MR,"FLAIR, T1w, t1gd, T2w"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4364,Colon,Tr,colon_215,Task10_Colon/labelsTr/colon_215.nii.gz,mask,Task10_Colon_Tr_colon_215,Mask,5f4eee388d796a497e4841cf951e4105a3375e19,"(512, 512, 92)",3,...,1.865516,1.293768,11.920381,1785.624593,"(14.437834281854327, 26.934006295205407, 34.84...",1.0,Task10_Colon/imagesTr/colon_215.nii.gz,colon cancer primaries,CT,
4365,Colon,Tr,colon_216,Task10_Colon/labelsTr/colon_216.nii.gz,mask,Task10_Colon_Tr_colon_216,Mask,9abb6686934896f2daf7aa2e3a1e900e4312d06f,"(512, 512, 90)",3,...,1.153354,1.660490,14.843588,2768.774925,"(22.795466117879528, 26.291231368785436, 43.65...",1.0,Task10_Colon/imagesTr/colon_216.nii.gz,colon cancer primaries,CT,
4366,Colon,Tr,colon_217,Task10_Colon/labelsTr/colon_217.nii.gz,mask,Task10_Colon_Tr_colon_217,Mask,e90c69f2cf0a1c76ffafdeec68248a00365e8946,"(512, 512, 96)",3,...,1.562260,1.304591,12.836191,2070.533167,"(17.450414466292635, 27.262092487174424, 35.56...",1.0,Task10_Colon/imagesTr/colon_217.nii.gz,colon cancer primaries,CT,
4367,Colon,Tr,colon_218,Task10_Colon/labelsTr/colon_218.nii.gz,mask,Task10_Colon_Tr_colon_218,Mask,8977e7e84fa67c9b4377ee9af743b51281795c17,"(512, 512, 90)",3,...,1.581470,1.457455,15.005894,2829.655908,"(19.500820073404867, 30.839959961567512, 44.94...",1.0,Task10_Colon/imagesTr/colon_218.nii.gz,colon cancer primaries,CT,


In [10]:
validate_index(index, "nifti", lazy=True)

,BodyPartExamined,split,ImageID,filepath,file_type,SampleID,class,hash,size,ndim,...,mask.flatness,mask.elongation,mask.equivalent_spherical_radius,mask.equivalent_spherical_perimeter,mask.equivalent_ellipsoid_diameters,mask.volume_count,reference_scan,ROINames,Modality,SeriesDescription
0,BrainTumour,Tr,BRATS_001,Task01_BrainTumour/imagesTr/BRATS_001.nii.gz,scan,Task01_BrainTumour_Tr_BRATS_001,MedImage,3d3c950a77051659e04b693871852b00f4ba7304,"(240, 240, 620)",3,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"edema, non-enhancing tumor, enhancing tumour",MR,"FLAIR, T1w, t1gd, T2w"
1,BrainTumour,Tr,BRATS_002,Task01_BrainTumour/imagesTr/BRATS_002.nii.gz,scan,Task01_BrainTumour_Tr_BRATS_002,MedImage,796d64011be8fcaf66f5ac8e26f27cb63847f882,"(240, 240, 620)",3,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"edema, non-enhancing tumor, enhancing tumour",MR,"FLAIR, T1w, t1gd, T2w"
2,BrainTumour,Tr,BRATS_003,Task01_BrainTumour/imagesTr/BRATS_003.nii.gz,scan,Task01_BrainTumour_Tr_BRATS_003,MedImage,fcbae1fba75fe6457940ac8beb4c9dcfa2dc29be,"(240, 240, 620)",3,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"edema, non-enhancing tumor, enhancing tumour",MR,"FLAIR, T1w, t1gd, T2w"
3,BrainTumour,Tr,BRATS_004,Task01_BrainTumour/imagesTr/BRATS_004.nii.gz,scan,Task01_BrainTumour_Tr_BRATS_004,MedImage,d54bccbe4ca6df61b4c7390011a496c11aebec3c,"(240, 240, 620)",3,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"edema, non-enhancing tumor, enhancing tumour",MR,"FLAIR, T1w, t1gd, T2w"
4,BrainTumour,Tr,BRATS_005,Task01_BrainTumour/imagesTr/BRATS_005.nii.gz,scan,Task01_BrainTumour_Tr_BRATS_005,MedImage,2f0ceff49b518db9a4d96c11454ade1ea1516675,"(240, 240, 620)",3,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"edema, non-enhancing tumor, enhancing tumour",MR,"FLAIR, T1w, t1gd, T2w"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4364,Colon,Tr,colon_215,Task10_Colon/labelsTr/colon_215.nii.gz,mask,Task10_Colon_Tr_colon_215,Mask,5f4eee388d796a497e4841cf951e4105a3375e19,"(512, 512, 92)",3,...,1.865516,1.293768,11.920381,1785.624593,"(14.437834281854327, 26.934006295205407, 34.84...",1.0,Task10_Colon/imagesTr/colon_215.nii.gz,colon cancer primaries,CT,
4365,Colon,Tr,colon_216,Task10_Colon/labelsTr/colon_216.nii.gz,mask,Task10_Colon_Tr_colon_216,Mask,9abb6686934896f2daf7aa2e3a1e900e4312d06f,"(512, 512, 90)",3,...,1.153354,1.660490,14.843588,2768.774925,"(22.795466117879528, 26.291231368785436, 43.65...",1.0,Task10_Colon/imagesTr/colon_216.nii.gz,colon cancer primaries,CT,
4366,Colon,Tr,colon_217,Task10_Colon/labelsTr/colon_217.nii.gz,mask,Task10_Colon_Tr_colon_217,Mask,e90c69f2cf0a1c76ffafdeec68248a00365e8946,"(512, 512, 96)",3,...,1.562260,1.304591,12.836191,2070.533167,"(17.450414466292635, 27.262092487174424, 35.56...",1.0,Task10_Colon/imagesTr/colon_217.nii.gz,colon cancer primaries,CT,
4367,Colon,Tr,colon_218,Task10_Colon/labelsTr/colon_218.nii.gz,mask,Task10_Colon_Tr_colon_218,Mask,8977e7e84fa67c9b4377ee9af743b51281795c17,"(512, 512, 90)",3,...,1.581470,1.457455,15.005894,2829.655908,"(19.500820073404867, 30.839959961567512, 44.94...",1.0,Task10_Colon/imagesTr/colon_218.nii.gz,colon cancer primaries,CT,


In [11]:
from pathlib import Path

new_index_dir = Path("/home/gpudual/bhklab/josh/med-image_index/new_index")
collection_path = new_index_dir / "MedicalDecathalon"
collection_path.mkdir(parents=True, exist_ok=True)

index.to_csv(collection_path / "index.csv", index=False)

In [ ]:
import json
source = indexed_datasets.source_config("MedicalDecathalon")

with open(collection_path / "source.json", "w") as f:
    json.dump(source.model_dump(mode="json"), f, indent=2)